# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, accessing fields and record sets by their `@id`.

### Dataset Source
FAIR² is provided as a [Croissant schema JSON-LD](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and contains multiple record sets, fields, and clinical variables for secondary analysis.

In [ ]:
# Ensure mlcroissant is available in this environment
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # mlcroissant returns a Metadata object

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"DOI/Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

We'll enumerate all record sets and for each, list available fields and columns, referencing them by their `@id`.


In [ ]:
# List all record sets by @id
print("Available record sets (@id):")
for record_set in dataset.record_sets:
    print(f"  - {record_set['@id']}")

# Let's inspect the first record set and enumerate its fields/columns by @id
if dataset.record_sets:
    # pick first record set
    first_record_set = dataset.record_sets[0]
    print(f"\nExploring record set '{first_record_set['@id']}' fields and columns:")
    for field in first_record_set['field']:
        print(f"  Field: {field['@id']}")
        if 'column' in field:
            print("    Columns:")
            for col in field['column']:
                print(f"      - {col['@id']}")
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from all available record sets into Pandas DataFrames. All record sets and fields are referenced by their `@id`.

In [ ]:
# Get all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from '{record_set_id}'.")
    except Exception as e:
        print(f"Failed to load '{record_set_id}': {e}")

# Display columns of the main record set (first one, by @id)
if record_set_ids:
    main_rsid = record_set_ids[0]
    main_df = dataframes[main_rsid]
    print(f"\nColumns for record set '{main_rsid}':")
    print(main_df.columns.tolist())
    main_df.head()
else:
    print('No record sets available.')

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps, using only `@id` references for fields and columns. We'll select a numeric field, perform filtering, normalization, and group-by operations.

In [ ]:
# Pick a numeric field from the first record set
# We demonstrate on column '@id's (e.g., age or interval fields if available)
# You should substitute these IDs according to your own dataset structure

# For demonstration, let's list columns to pick a numeric one
main_rsid = record_set_ids[0]
df = dataframes[main_rsid]
print("DataFrame columns and sample values:")
print(df.head())

# Try to automatically pick a numeric column
numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) == 0:
    print("No numeric columns identified.")
else:
    numeric_field_id = numeric_cols[0]   # choose the first numeric column by @id
    print(f"Using numeric field @id: '{numeric_field_id}'")
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 0
    filtered_df = df[df[numeric_field_id].notnull() & (df[numeric_field_id] > threshold)]
    print(f"Filtered records in '{main_rsid}' where {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df[[numeric_field_id]].head())
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized '{numeric_field_id}' values:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by a non-numeric/grouping field if present: search for object columns
    group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
    if group_fields:
        group_field_id = group_fields[0]
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
        print(f"\nMean {numeric_field_id} grouped by '{group_field_id}':")
        print(grouped.head())
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization

Visualize the distribution of the selected numeric field and relationship between two fields (if possible). All fields referenced by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_cols) == 0:
    print("No numeric columns to plot.")
else:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If there is also a groupable field, show a boxplot
    if group_fields:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Boxplot of '{numeric_field_id}' grouped by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook demonstrated step-by-step exploration of the Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer dataset via the `mlcroissant` library.
- All record sets, fields, and columns were accessed by their `@id` for reproducible, schema-driven processing.
- You can further extend the EDA and modeling by leveraging the Croissant schema metadata, ensuring all field accesses use the canonical `@id` references.